In [ ]:
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D

import os
import pandas as pd
import numpy as np

from keras.models import model_from_json
from tqdm.notebook import tqdm
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
%matplotlib inline

ModuleNotFoundError: No module named 'tensorflow'

In [5]:
TRAIN_DIR = 'archive/images/train'
TEST_DIR = 'archive/images/test'

In [6]:
def createdataframe(dir):
    image_paths = []
    labels = []
    for label in os.listdir(dir):
        for imagename in os.listdir(os.path.join(dir,label)):
            image_paths.append(os.path.join(dir,label,imagename))
            labels.append(label)
        print(label, "completed")
    return image_paths,labels


In [7]:
train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [8]:
print(train)

                                        image     label
0            archive/images/train\angry\0.jpg     angry
1            archive/images/train\angry\1.jpg     angry
2           archive/images/train\angry\10.jpg     angry
3        archive/images/train\angry\10002.jpg     angry
4        archive/images/train\angry\10016.jpg     angry
...                                       ...       ...
28816  archive/images/train\surprise\9969.jpg  surprise
28817  archive/images/train\surprise\9985.jpg  surprise
28818  archive/images/train\surprise\9990.jpg  surprise
28819  archive/images/train\surprise\9992.jpg  surprise
28820  archive/images/train\surprise\9996.jpg  surprise

[28821 rows x 2 columns]


In [11]:
test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)

angry completed
disgust completed
fear completed
happy completed
neutral completed
sad completed
surprise completed


In [ ]:
print(test)

                                      image     label
0       archive/images/test\angry\10052.jpg     angry
1       archive/images/test\angry\10065.jpg     angry
2       archive/images/test\angry\10079.jpg     angry
3       archive/images/test\angry\10095.jpg     angry
4       archive/images/test\angry\10121.jpg     angry
...                                     ...       ...
7061  archive/images/test\surprise\9806.jpg  surprise
7062  archive/images/test\surprise\9830.jpg  surprise
7063  archive/images/test\surprise\9853.jpg  surprise
7064  archive/images/test\surprise\9878.jpg  surprise
7065   archive/images/test\surprise\993.jpg  surprise

[7066 rows x 2 columns]


In [14]:
print(test['image'])

0         archive/images/test\angry\10052.jpg
1         archive/images/test\angry\10065.jpg
2         archive/images/test\angry\10079.jpg
3         archive/images/test\angry\10095.jpg
4         archive/images/test\angry\10121.jpg
                        ...                  
7061    archive/images/test\surprise\9806.jpg
7062    archive/images/test\surprise\9830.jpg
7063    archive/images/test\surprise\9853.jpg
7064    archive/images/test\surprise\9878.jpg
7065     archive/images/test\surprise\993.jpg
Name: image, Length: 7066, dtype: str


In [13]:
from tqdm.notebook import tqdm

In [ ]:
def extract_features(images):
    features = []
    for image in tqdm(images):
        img = load_img(image,color_mode='grayscale' )
        img = np.array(img)
        features.append(img)

    
    features = features.reshape(len(features),48,48,1)  #doubt
    return features
    

In [19]:
train_features = extract_features(train['image'])

  0%|          | 0/28821 [00:00<?, ?it/s]

In [20]:
test_features = extract_features(test['image'])

  0%|          | 0/7066 [00:00<?, ?it/s]

In [21]:
x_train = train_features/255.0
x_test = test_features/255.0

In [22]:
from sklearn.preprocessing import LabelEncoder

In [23]:
le = LabelEncoder()
le.fit(train['label'])

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](7,)","['angry','disgust','fear',...,'neutral','sad','surprise']"


In [24]:
y_train = le.transform(train['label'])
y_test = le.transform(test['label'])

In [25]:
y_train = to_categorical(y_train,num_classes = 7)
y_test = to_categorical(y_test,num_classes = 7)

In [26]:
model = Sequential()
# convolutional layers
model.add(Conv2D(128, kernel_size=(3,3), activation='relu', input_shape=(48,48,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(256, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Conv2D(512, kernel_size=(3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.4))

model.add(Flatten())
# fully connected layers
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.4))
model.add(Dense(256, activation='relu'))
model.add(Dropout(0.3))
# output layer
model.add(Dense(7, activation='softmax'))

C:\Users\surya_t4rpfms\AppData\Roaming\Python\Python314\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [29]:
model.fit(x= x_train,y = y_train, batch_size = 128, epochs = 100, validation_data = (x_test,y_test)) 

Epoch 1/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 436s 2s/step - accuracy: 0.2442 - loss: 1.8249 - val_accuracy: 0.2583 - val_loss: 1.8090
Epoch 2/100
226/226 ━━━━━━━━━━━━━━━━━━━━ 359s 2s/step - accuracy: 0.2479 - loss: 1.8097 - val_accuracy: 0.2603 - val_loss: 1.7857
Epoch 3/100
182/226 ━━━━━━━━━━━━━━━━━━━━ 1:10 2s/step - accuracy: 0.2658 - loss: 1.7710

KeyboardInterrupt: 

In [ ]:
label = ['angry','disgust','fear','happy','neutral','sad','surprise']

In [ ]:
def ef(image):
    img = load_img(image,grayscale =  True )
    feature = np.array(img)
    feature = feature.reshape(1,48,48,1)
    return feature/255.0
    

In [ ]:
image = 'archive/images/train/sad/42.jpg'
print("original image is of sad")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)

In [ ]:
image = 'archive/images/train/sad/42.jpg'
print("original image is of sad")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)
plt.imshow(img.reshape(48,48),cmap='gray')

In [ ]:
image = 'archive/images/train/happy/7.jpg'
print("original image is of happy")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)
plt.imshow(img.reshape(48,48),cmap='gray')

In [ ]:
image = 'archive/images/train/surprise/15.jpg'
print("original image is of surprise")
img = ef(image)
pred = model.predict(img)
pred_label = label[pred.argmax()]
print("model prediction is ",pred_label)
plt.imshow(img.reshape(48,48),cmap='gray')